In [17]:
import torch 
import torch.nn as nn
import matplotlib.pyplot as plt

In [18]:
from torchvision.datasets import CIFAR10
from torchvision import transforms

transform = transforms.ToTensor()

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        (0.5, 0.5, 0.5 ),
        (0.5, 0.5, 0.5 )
    )
])

train_dataset = CIFAR10(
    root="./datasets",
    train=True,
    download=True,
    transform=transform
)

test_dataset = CIFAR10(
    root="./datasets",
    train=False,
    download=True,
    transform=transform
)

print(len(train_dataset))
print(len(test_dataset))

50000
10000


In [19]:
torch.randperm(10)

tensor([4, 7, 6, 5, 8, 9, 2, 0, 1, 3])

In [20]:
img = []
for i in range(10):

    img.append(train_dataset[i][0])

len(img)

10

In [21]:
# fig, ax  = plt.subplots(2, 5, figsize=(5, 5))
# for i, image in zip(ax.flat, img): 
#     i.imshow(image)
#     i.axis("off")
# plt.tight_layout()
# plt.show()


In [22]:
print(type(train_dataset))
print(train_dataset.classes)

<class 'torchvision.datasets.cifar.CIFAR10'>
['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']


In [23]:
f = [0] * 10

for i, label in enumerate(train_dataset.targets):
    f[label] += 1 
print(f)

[5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000]


In [24]:
class myCNN(nn.Module):

    def __init__(self):

        super().__init__()

        self.conv1 = nn.Conv2d(in_channels=3, out_channels=8, kernel_size=5)

        self.maxPool = nn.MaxPool2d(kernel_size=2)

        self.conv2 = nn.Conv2d(in_channels=8, out_channels=16, kernel_size=3)

        self.avgPool = nn.AvgPool2d(kernel_size=2)

        self.fc1 = nn.Linear(576, 28)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(28, 10)


    def forward(self, X):

        x = X
        x = self.conv1(x)
        x = self.relu(x)
        x = self.maxPool(x)
        x = self.conv2(x)
        x = self.relu(x)
        x = self.avgPool(x) 
        x = self.fc1(torch.flatten(x, start_dim=1))
        x = self.relu(x)
        x = self.fc2(x) 

        return x


In [25]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)

In [26]:
model = myCNN()

optimzer = torch.optim.SGD(lr = 0.01, params=model.parameters())

loss_fn = nn.CrossEntropyLoss()


In [27]:
epoch = 10 

for i in range(epoch):

    current_loss = 0 
    for images, label in train_loader:

        output = model(images)
        loss = loss_fn(output, label)

        optimzer.zero_grad()
        loss.backward()
        optimzer.step()
        

        current_loss += loss 
    
    loss = current_loss/ len(train_loader)

    print(f"Epoch : {i+1}  || Loss : {loss}")
        

Epoch : 1  || Loss : 2.2015836238861084
Epoch : 2  || Loss : 1.9715287685394287
Epoch : 3  || Loss : 1.8063900470733643
Epoch : 4  || Loss : 1.6235060691833496
Epoch : 5  || Loss : 1.53033447265625
Epoch : 6  || Loss : 1.4614728689193726
Epoch : 7  || Loss : 1.408302903175354
Epoch : 8  || Loss : 1.371787190437317
Epoch : 9  || Loss : 1.3382987976074219
Epoch : 10  || Loss : 1.3113257884979248


In [30]:
test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False
)

In [31]:
model.eval()

correct = 0
total = 0

with torch.no_grad():

    for images, labels in test_loader:

        outputs = model(images)

        predictions = outputs.argmax(dim=1)

        correct += (predictions == labels).sum().item()

        total += labels.size(0)

accuracy = 100 * correct / total

print(f"Test Accuracy: {accuracy:.2f}%")

Test Accuracy: 51.82%
